In [ ]:
import asyncio
import os
import sqlite3
from pathlib import Path
from typing import Any

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    SystemMessage,
    create_sdk_mcp_server,
    query,
    tool,
)
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel


if os.name == "nt":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


BASE_DIR = find_workspace_root(Path.cwd())
DATA_DIR = BASE_DIR / "data"
DB_PATH = DATA_DIR / "module4_sales.sqlite"
NOTEBOOK_DIR = BASE_DIR / "notebooks"
REPORT_DIR = NOTEBOOK_DIR / "report"
REPORT_PATH = REPORT_DIR / "module4_sqlite_report.md"

TASK = f'''
Use the SQLite database exposed through MCP to produce a concise markdown report that answers these questions:
1. Which region generated the most revenue?
2. What are the top 3 products by revenue?
3. What short business insight would you give a manager based on those results?

Before answering, inspect the database schema if needed and use only the MCP database tool.
'''


def seed_database() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("PRAGMA foreign_keys = ON")
        connection.executescript("""
            DROP TABLE IF EXISTS order_items;
            DROP TABLE IF EXISTS orders;
            DROP TABLE IF EXISTS customers;
            DROP TABLE IF EXISTS products;

            CREATE TABLE customers (
                customer_id INTEGER PRIMARY KEY,
                customer_name TEXT NOT NULL,
                region TEXT NOT NULL
            );

            CREATE TABLE products (
                product_id INTEGER PRIMARY KEY,
                product_name TEXT NOT NULL,
                category TEXT NOT NULL,
                unit_price REAL NOT NULL
            );

            CREATE TABLE orders (
                order_id INTEGER PRIMARY KEY,
                customer_id INTEGER NOT NULL,
                order_date TEXT NOT NULL,
                FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
            );

            CREATE TABLE order_items (
                order_item_id INTEGER PRIMARY KEY,
                order_id INTEGER NOT NULL,
                product_id INTEGER NOT NULL,
                quantity INTEGER NOT NULL,
                FOREIGN KEY (order_id) REFERENCES orders(order_id),
                FOREIGN KEY (product_id) REFERENCES products(product_id)
            );
        """)

        customers = [
            (1, "Avery Chen", "North"),
            (2, "Nadia Patel", "South"),
            (3, "Marcus Reed", "East"),
            (4, "Elena Torres", "West"),
        ]
        products = [
            (1, "Analytics Seat", "Software", 120.0),
            (2, "Support Pack", "Service", 80.0),
            (3, "Training Workshop", "Service", 150.0),
            (4, "Monitoring Add-on", "Software", 95.0),
        ]
        orders = [
            (1, 1, "2026-04-02"),
            (2, 2, "2026-04-05"),
            (3, 3, "2026-04-06"),
            (4, 4, "2026-04-10"),
            (5, 1, "2026-04-12"),
            (6, 2, "2026-04-15"),
            (7, 3, "2026-04-18"),
            (8, 4, "2026-04-20"),
        ]
        order_items = [
            (1, 1, 1, 2),
            (2, 1, 2, 1),
            (3, 2, 3, 1),
            (4, 2, 4, 2),
            (5, 3, 1, 1),
            (6, 3, 3, 2),
            (7, 4, 2, 3),
            (8, 4, 4, 1),
            (9, 5, 1, 3),
            (10, 6, 2, 2),
            (11, 7, 3, 1),
            (12, 8, 4, 4),
        ]

        connection.executemany(
            "INSERT INTO customers (customer_id, customer_name, region) VALUES (?, ?, ?)",
            customers,
        )
        connection.executemany(
            "INSERT INTO products (product_id, product_name, category, unit_price) VALUES (?, ?, ?, ?)",
            products,
        )
        connection.executemany(
            "INSERT INTO orders (order_id, customer_id, order_date) VALUES (?, ?, ?)",
            orders,
        )
        connection.executemany(
            "INSERT INTO order_items (order_item_id, order_id, product_id, quantity) VALUES (?, ?, ?, ?)",
            order_items,
        )
        connection.commit()


def render_markdown_table(headers: list[str], rows: list[sqlite3.Row]) -> str:
    if not rows:
        return "Query returned no rows."

    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        values = []
        for header in headers:
            value = row[header]
            if isinstance(value, float):
                values.append(f"{value:.2f}")
            elif value is None:
                values.append("")
            else:
                values.append(str(value))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


@tool(
    "query_sql",
    "Run a read-only SQL query against the local SQLite analytics database.",
    {"sql": str},
)
async def query_sql(arguments: dict[str, Any]) -> dict[str, Any]:
    sql = arguments["sql"].strip()
    normalized = sql.lower().lstrip()
    if not normalized.startswith(("select", "with", "pragma", "explain")):
        return {
            "content": [
                {
                    "type": "text",
                    "text": "Only read-only SELECT, WITH, PRAGMA, or EXPLAIN statements are allowed.",
                }
            ],
            "is_error": True,
        }

    stripped = sql.rstrip().rstrip(";")
    if ";" in stripped:
        return {
            "content": [
                {
                    "type": "text",
                    "text": "Only single-statement queries are allowed.",
                }
            ],
            "is_error": True,
        }

    with sqlite3.connect(DB_PATH) as connection:
        connection.row_factory = sqlite3.Row
        cursor = connection.execute(sql)
        if cursor.description is None:
            return {
                "content": [
                    {
                        "type": "text",
                        "text": "Query executed successfully but returned no tabular results.",
                    }
                ]
            }

        headers = [column[0] for column in cursor.description]
        rows = cursor.fetchmany(25)
        table_text = render_markdown_table(headers, rows)
        return {
            "content": [
                {
                    "type": "text",
                    "text": table_text,
                }
            ]
        }


seed_database()

console = Console()
console.print(Panel("[bold white]Agent SDK + MCP + SQLite[/bold white]", expand=False))
console.print(f"[cyan]SQLite database ready:[/cyan] {DB_PATH}")

╭──────────────────────────╮
│ Agent SDK + MCP + SQLite │
╰──────────────────────────╯

SQLite database ready: d:\stuff\CogKnowEdge Solutions\new\SDK\Module1\data\module4_sales.sqlite

In [11]:
load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY environment variable is not set.")

sqlite_server = create_sdk_mcp_server(
    name="sqlite",
    version="1.0.0",
    tools=[query_sql],
)

options = ClaudeAgentOptions(
    mcp_servers={"sqlite": sqlite_server},
    allowed_tools=["mcp__sqlite__query_sql"],
    model="claude-haiku-4-5-20251001",
)

console.print(f"[cyan]Allowed MCP tools:[/cyan] {options.allowed_tools}")

Allowed MCP tools: ['mcp__sqlite__query_sql']

In [12]:
async def run_database_analyst(task_prompt: str, agent_options: ClaudeAgentOptions) -> str:
    final_output = ""

    try:
        async for message in query(prompt=task_prompt, options=agent_options):
            if isinstance(message, SystemMessage) and message.subtype == "init":
                console.print("[bold blue]Agent session initialized.[/bold blue]")
                console.print(message.data.get("mcp_servers"))

            if isinstance(message, AssistantMessage):
                tool_calls = [
                    block.name
                    for block in message.content
                    if hasattr(block, "type") and block.type == "tool_use"
                ]
                if tool_calls:
                    console.print(f"[dim]Tool calls requested: {', '.join(tool_calls)}[/dim]")

            if isinstance(message, ResultMessage):
                if message.subtype == "success":
                    final_output = message.result or ""
                    console.print("[bold green]Execution completed successfully.[/bold green]")
                    break
                final_output = f"Execution stopped with status: {message.subtype}"
                console.print(f"[bold red]Execution stopped: {message.subtype}[/bold red]")
                break
    except Exception as exc:
        if final_output and "Claude Code returned an error result: success" in str(exc):
            return final_output
        message = str(exc)
        if "Failed to authenticate" in message or "API Error: 403" in message:
            return (
                "Authentication failed while calling the model API. "
                "Check ANTHROPIC_API_KEY, account access, and model availability."
            )
        raise

    return final_output

In [ ]:
import asyncio

REQUEST_TIMEOUT_SECONDS = 120


def run_database_analyst_sync() -> str:
    return asyncio.run(
        asyncio.wait_for(run_database_analyst(TASK, options), timeout=REQUEST_TIMEOUT_SECONDS)
    )


try:
    response_text = await asyncio.to_thread(run_database_analyst_sync)
except asyncio.TimeoutError:
    response_text = (
        f"Execution timed out after {REQUEST_TIMEOUT_SECONDS} seconds before the model returned a final report."
    )
    console.print(f"[bold red]{response_text}[/bold red]")
except Exception as exc:
    response_text = f"Execution failed: {exc}"
    console.print(f"[bold red]{response_text}[/bold red]")

console.print(Panel(Markdown(response_text or "No report was generated."), title="[bold green]Agent Response[/bold green]"))

REPORT_DIR.mkdir(parents=True, exist_ok=True)
with open(REPORT_PATH, "w", encoding="utf-8") as report_file:
    report_file.write(response_text)

console.print(f"[bold cyan]Report saved to disk:[/bold cyan] {REPORT_PATH}")

Agent session initialized.

[{'name': 'sqlite', 'status': 'connected'}]

Execution completed successfully.

an error occurred during closing of asynchronous generator <async_generator object InternalClient._process_query_inner at 0x00000179598BD430>
asyncgen: <async_generator object InternalClient._process_query_inner at 0x00000179598BD430>
RuntimeError: aclose(): asynchronous generator is already running


╭──────────────────────────────────────────────── Agent Response ─────────────────────────────────────────────────╮
│                                            Database Analysis Report                                             │
│                                                                                                                 │
│ 1. Region with Most Revenue                                                                                     │
│                                                                                                                 │
│ West generated the most revenue at $715.00, followed by North ($680.00), East ($570.00), and South ($500.00).   │
│                                                                                                                 │
│ 2. Top 3 Products by Revenue                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  Product            Revenue                                                                                     │
│  ──────────────────────────                                                                                     │
│  Analytics Seat     $720.00                                                                                     │
│  Monitoring Add-on  $665.00                                                                                     │
│  Training Workshop  $600.00                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│ 3. Business Insight                                                                                             │
│                                                                                                                 │
│ Focus on the West region while strengthening underperforming markets. The West region generates nearly 43% more │
│ revenue than the South, and the top three products (Analytics Seat, Monitoring Add-on, and Training Workshop)   │
│ account for $1,985 of the total $2,465 in revenue—81% of all sales. This suggests an opportunity to apply       │
│ successful West region strategies to the South and East regions, while investigating whether these              │
│ top-performing products are driving growth or if there's untapped demand in lagging markets.                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Report saved to disk: d:\stuff\CogKnowEdge Solutions\new\SDK\Module1\module4_sqlite_report.md